# Select a configuration and evaluate it

## Goal and setup

Compare three Ridge configurations on development rows, then fit the chosen
configuration freshly and inspect an untouched holdout. This uses 24 artificial
fixtures and synthetic continuous outcomes. No predictive performance claim is
made. Use the existing Python (misc314) kernel.

### 1. Prepare data and two separate scopes

Rows 0–17 are development; rows 18–23 are the untouched holdout. Each inner fold
has disjoint training/test rows. Score rows determine the selection metric.
For an actual project, start from your assembled ModelDataset and prepared split
plan; retain its original row order.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
sys.path.insert(0, str(root / "src"))
from xdiyo_analytics.datasets import ModelDataset
from xdiyo_analytics.splits import Fold, SplitPlan
from xdiyo_analytics.analysis import PreTrainingAnalysis, PostTrainingAnalysis
from xdiyo_analytics.reporting import TopKCorrelationSelector, PerformanceReporter, MatchResultReporter
from xdiyo_analytics.training import EstimatorAdapter
from xdiyo_analytics.selection import Candidate, GridCandidates, ModelSelection
from xdiyo_analytics.experiments import ExperimentStore
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

rows = np.arange(24)
X = pd.DataFrame({"signal": np.sin(rows / 3) + rows / 25, "noise": np.cos(rows * 1.7)})
y = pd.DataFrame({"total": 8 + 2 * X.signal + .1 * np.cos(rows * 2)})
metadata = pd.DataFrame({
    "competition_id": 1, "season_id": 2026, "event_id": rows + 1001,
    "round": rows // 4 + 1, "home_id": 1 + rows % 3, "away_id": 4 + rows % 3,
    "home_name": ["Home " + str(i % 3 + 1) for i in rows],
    "away_name": ["Away " + str(i % 3 + 4) for i in rows],
})
keys = ("competition_id", "season_id", "event_id")
dataset = ModelDataset(X, y, metadata, "match", keys, keys, "total")
development = np.arange(18)
inner_plan = SplitPlan([
    Fold(np.arange(8), np.arange(8, 12), np.arange(8, 12)),
    Fold(np.arange(12), np.arange(12, 18), np.arange(12, 18)),
], len(X), rows)
holdout = Fold(development, np.arange(18, 24), np.arange(18, 24))
output_dir = root / "experiment/model_selection_demo/notebook"
output_dir.mkdir(parents=True, exist_ok=True)

### 2. Compare fresh candidate fits

Each factory creates a new scaler and Ridge estimator. Feature selection learns
one column separately from each fitting population. The overall MSE is one
calculation over pooled inner score predictions; these folds have different
sizes. Search stores three trial records. It does not save a final evaluation.

In [2]:
preparation = PreTrainingAnalysis({
    "columns": TopKCorrelationSelector(type="per_fold", partition="train", k=1, method="pearson"),
})

def candidate(parameters):
    alpha = parameters["alpha"]
    return Candidate(
        "Ridge", lambda: EstimatorAdapter(make_pipeline(StandardScaler(), Ridge(alpha=alpha))),
        config={"model": "Ridge", "alpha": alpha, "scaler": "StandardScaler",
                "features": {"selector": "absolute Pearson", "k": 1}},
        pre_analysis=preparation, features_from="columns",
    )

store = ExperimentStore(output_dir / "experiments", "Synthetic notebook selection")
selection = ModelSelection(
    GridCandidates({"alpha": [.01, .2, 2.]}, candidate), metrics="mse",
).run(dataset, inner_plan, development_positions=development, experiment=store)
print("Selected:", selection.winner.candidate.config)
print("Inner fits per candidate:", [len(trial.training.folds) for trial in selection.trials])

Selected: {'model': 'Ridge', 'alpha': 0.01, 'scaler': 'StandardScaler', 'features': {'selector': 'absolute Pearson', 'k': 1}, 'grid_parameters': {'alpha': 0.01}}
Inner fits per candidate: [2, 2, 2]


### 3. Inspect the chosen configuration

The table retains raw MSE, normalized utility, rank and the selected flag.
Its scope export lists all 18 declared development rows. This comparison is
internal selection evidence; independent evaluation follows in the next cell.

Saved HTML and native iframe interactions are verified. Live Jupyter frontend
trust/display behavior remains unverified.

In [3]:
comparison = selection.to_report()
comparison.to_html(output_dir / "comparison.html")
comparison.to_notebook(height=800)

### 4. Evaluate the untouched holdout and save it explicitly

The selected feature rule, scaler and model are fitted again on development.
The report displays six holdout predictions and their observed outcomes. The
separate final record links to the winner's saved trial ID. The default
experiment leaderboard shows final records; internal trials are optional.

The fixture tolerance of 0.5 is a display rule, not the model-selection
criterion. Green cells mean absolute error at most 0.5.

In [4]:
evaluation = selection.evaluate(dataset, holdout)
report = PostTrainingAnalysis({
    "Holdout errors": PerformanceReporter(type="overall", partition="score", metrics=["mse", "mae"]),
    "Holdout fixtures": MatchResultReporter(type="per_fold", target="total", tolerance=.5),
}, title="Synthetic untouched holdout").run(evaluation)
saved = store.save_run(
    evaluation, report, name="Untouched holdout", role="final",
    config=selection.winner.candidate.config, run_group=selection.run_group,
    selected_trial_id=selection.winner.saved_run_id,
)
print("Fresh holdout fit; selected columns:", evaluation.folds[0].feature_columns)
print("Saved roles:", [record["role"] for record in store.read_runs()])
report.to_html(output_dir / "holdout.html")
report.to_notebook(height=900)

Fresh holdout fit; selected columns: ('signal',)
Saved roles: ['trial', 'trial', 'trial', 'final']


## Next steps

The [guide](../docs/analytics/model_selection.md) adds conditional
Ridge/Lasso/ElasticNet grids, weighted/parsimony decisions, optional nested CV
and justified fitted likelihood evidence. See the
[reference](../docs/analytics/model_selection_reference.md),
[equations](../docs/analytics/model_selection_equations.md),
[coverage](../docs/analytics/model_selection_documentation_checklist.md) and
[verification record](../docs/analytics/model_selection_check.json).
The preceding twelve notebooks and their outputs remain unchanged.